# Model sederhana vs model lengkap — hs-CRP IFLS-5

Notebook ini menjawab pertanyaan yang hampir pasti diajukan reviewer: **apa tambahan model ML lengkap (32 prediktor) dibanding model logistik sederhana?**

Dua model sederhana dibandingkan dengan ensemble dan XGBoost:
- **LR: BMI saja**
- **LR: 5 butir** — BMI, jenis kelamin, umur, kesehatan dirasa buruk, tempat tinggal (semua bisa ditanyakan/diukur di Posbindu tanpa tes darah)

Pembagian data latih/uji dan fold validasi silang **identik** dengan `analisis_tambahan.ipynb` (seed 42). Hasil disimpan di `results/analisis_model_sederhana.xlsx`. Waktu jalan sekitar 5–10 menit.

In [ ]:
# === SEL 0: folder kerja dan file data ===
import os
ROOT = r"F:\SINTA 3\crp-ifls5-ml"
OUTDIR = os.path.join(ROOT, "results")
os.makedirs(OUTDIR, exist_ok=True)
os.chdir(OUTDIR)
DATA_FILE = os.path.join(ROOT, "data", "crp_ifls5_model.csv")
ANALYTIC_FILE = r"F:\SINTA 3\olahan_ifls5\ifls5_analytic.csv"   # untuk analisis CRP > 10 dan perbandingan peserta dikeluarkan
assert os.path.exists(DATA_FILE), f"Data tidak ditemukan: {DATA_FILE}"
print("Folder kerja :", os.getcwd())
print("Data model   :", DATA_FILE)
print("Data analitik:", ANALYTIC_FILE, "(ada)" if os.path.exists(ANALYTIC_FILE) else "(TIDAK ADA - analisis 4a dan 4c dilewati)")

# Jumlah ulangan validasi silang. 5 = cukup untuk naskah (sekitar 30-60 menit dengan DNN). Naikkan ke 10 bila ada waktu.
REPEATS = 5
RUN_DNN = False   # DNN tidak diperlukan untuk perbandingan ini

In [ ]:
import warnings, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
from scipy import stats
from sklearn.model_selection import (train_test_split, StratifiedKFold, StratifiedGroupKFold, RepeatedStratifiedKFold,
                                     cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.base import clone, BaseEstimator, ClassifierMixin

def ada(m):
    try: __import__(m); return True
    except ImportError: return False
HAS_XGB, HAS_LGBM, HAS_TORCH = ada('xgboost'), ada('lightgbm'), ada('torch')
MODE = 'FINAL' if (HAS_XGB and HAS_LGBM) else 'PRATINJAU'
print(f'XGBoost={HAS_XGB} | LightGBM={HAS_LGBM} | PyTorch={HAS_TORCH} | MODE: {MODE}')
SEED = 42

In [ ]:
df = pd.read_csv(DATA_FILE, dtype={'pidlink': str, 'hhid14': str})
NUM = ['age', 'lnpce', 'hhsize', 'cig_day', 'pa_vig_days', 'pa_mod_days', 'pa_walk_days', 'bmi', 'sbp', 'dbp', 'n_chronic', 'cesd10']
BIN = ['female', 'married', 'widowed', 'urban', 'dx_hypert', 'dx_diab', 'dx_heart', 'dx_stroke', 'dx_chol',
       'dx_asthma', 'dx_lung', 'dx_arthritis', 'dx_kidney', 'dx_liver', 'dx_digest', 'med_hypert', 'med_diab', 'srh_poor']
CAT = ['educ', 'smoke_status']
FEATS = NUM + BIN + CAT
X, y, groups = df[FEATS].copy(), df.crp_high.values, df.hhid14.values

# Pembagian latih/uji IDENTIK dengan notebook utama (seed 42, stratified)
X_tr, X_te, y_tr, y_te, idx_tr, idx_te = train_test_split(X, y, df.index, test_size=0.2, stratify=y, random_state=SEED)
hh_tr, hh_te = set(df.loc[idx_tr, 'hhid14']), df.loc[idx_te, 'hhid14']
shared = hh_te.isin(hh_tr)
n_hh = df.hhid14.nunique(); size = df.groupby('hhid14').size()
print(f'n = {len(df)}, kasus = {y.sum()} ({y.mean()*100:.1f}%) | rumah tangga = {n_hh}; '
      f'{(size > 1).sum()} rumah tangga punya >1 peserta ({size[size > 1].sum()} peserta)')
print(f'Data uji: {shared.sum()} dari {len(idx_te)} peserta ({shared.mean()*100:.1f}%) punya anggota rumah tangga di data latih')

def prep(scale, feats=FEATS):
    num, bin_, cat = [c for c in NUM if c in feats], [c for c in BIN if c in feats], [c for c in CAT if c in feats]
    num_steps = [('imp', SimpleImputer(strategy='median'))] + ([('sc', StandardScaler())] if scale else [])
    return ColumnTransformer([('num', Pipeline(num_steps), num), ('bin', SimpleImputer(strategy='most_frequent'), bin_),
                              ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                                                ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat)],
                             verbose_feature_names_out=False)

In [ ]:
# === Model dengan hyperparameter terbaik dari notebook utama ===
class BalancedMLP(MLPClassifier):
    def fit(self, X, y, sample_weight=None):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y))

if HAS_TORCH:
    import torch, torch.nn as nn
    class TorchDNN(ClassifierMixin, BaseEstimator):
        def __init__(self, hidden=(64, 32), dropout=0.3, lr=1e-3, weight_decay=1e-5, epochs=300, batch=128, patience=25, seed=SEED):
            self.hidden, self.dropout, self.lr, self.weight_decay = hidden, dropout, lr, weight_decay
            self.epochs, self.batch, self.patience, self.seed = epochs, batch, patience, seed
        def _net(self, d):
            layers = []
            for h in self.hidden: layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(self.dropout)]; d = h
            return nn.Sequential(*layers, nn.Linear(d, 1))
        def fit(self, X, y):
            torch.manual_seed(self.seed); X = np.asarray(X, np.float32); y = np.asarray(y, np.float32)
            self.classes_ = np.array([0, 1])
            Xa, Xv, ya, yv = train_test_split(X, y, test_size=0.15, stratify=y, random_state=self.seed)
            self.net_ = self._net(X.shape[1]); opt = torch.optim.AdamW(self.net_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float((ya == 0).sum() / (ya == 1).sum())))
            Xa_t, ya_t, Xv_t = torch.tensor(Xa), torch.tensor(ya), torch.tensor(Xv)
            best, wait, state = -1, 0, None
            for ep in range(self.epochs):
                self.net_.train(); perm = torch.randperm(len(Xa_t))
                for i in range(0, len(perm), self.batch):
                    b = perm[i:i + self.batch]
                    if len(b) < 2: continue
                    opt.zero_grad(); lossf(self.net_(Xa_t[b]).squeeze(1), ya_t[b]).backward(); opt.step()
                self.net_.eval()
                with torch.no_grad(): auc = roc_auc_score(yv, self.net_(Xv_t).squeeze(1).numpy())
                if auc > best: best, wait, state = auc, 0, {k: v.clone() for k, v in self.net_.state_dict().items()}
                else:
                    wait += 1
                    if wait >= self.patience: break
            self.net_.load_state_dict(state); return self
        def predict_proba(self, X):
            self.net_.eval()
            with torch.no_grad(): p = torch.sigmoid(self.net_(torch.tensor(np.asarray(X, np.float32))).squeeze(1)).numpy()
            return np.c_[1 - p, p]
        def predict(self, X): return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

POS_W = (y_tr == 0).sum() / (y_tr == 1).sum()      # sama dengan notebook utama (5,62)
if HAS_XGB: from xgboost import XGBClassifier

def make_models(feats=FEATS):
    M = {}
    M['Logistic regression'] = Pipeline([('p', prep(True, feats)), ('m', LogisticRegression(C=0.011288378916846888, class_weight='balanced', max_iter=3000))])
    M['Random forest'] = Pipeline([('p', prep(False, feats)), ('m', RandomForestClassifier(n_estimators=500, max_depth=8, min_samples_leaf=40,
                                   max_features='sqrt', class_weight='balanced_subsample', n_jobs=-1, random_state=SEED))])
    if HAS_XGB:
        M['XGBoost'] = Pipeline([('p', prep(False, feats)), ('m', XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.01, subsample=0.85, colsample_bytree=0.8,
                                 min_child_weight=10, scale_pos_weight=POS_W, eval_metric='logloss', n_jobs=-1, random_state=SEED))])
    else:
        M['HistGB (pengganti XGBoost)'] = Pipeline([('p', prep(False, feats)), ('m', HistGradientBoostingClassifier(class_weight='balanced', max_iter=200,
                                                   learning_rate=0.03, max_leaf_nodes=7, random_state=SEED))])
    if HAS_LGBM:
        from lightgbm import LGBMClassifier
        M['LightGBM'] = Pipeline([('p', prep(False, feats)), ('m', LGBMClassifier(n_estimators=200, num_leaves=15, learning_rate=0.01, min_child_samples=50,
                                  subsample=0.7, colsample_bytree=0.6, class_weight='balanced', random_state=SEED, verbose=-1))])
    M['MLP'] = Pipeline([('p', prep(True, feats)), ('m', BalancedMLP(hidden_layer_sizes=(64, 32), alpha=1.0, learning_rate_init=0.003, batch_size=64,
                         early_stopping=True, validation_fraction=0.15, n_iter_no_change=20, max_iter=600, random_state=SEED))])
    if HAS_TORCH and RUN_DNN:
        M['DNN (PyTorch)'] = Pipeline([('p', prep(True, feats)), ('m', TorchDNN())])
    M['Ensemble (RF + MLP + LR)'] = VotingClassifier([('rf', clone(M['Random forest'])), ('mlp', clone(M['MLP'])),
                                                      ('lr', clone(M['Logistic regression']))], voting='soft')
    return M

MODELS = make_models()
print('Model:', list(MODELS))

In [ ]:
SIMPLE = {'LR: BMI only': ['bmi'],
          'LR: 5 items (BMI, sex, age, self-rated health, urban)': ['bmi', 'female', 'age', 'srh_poor', 'urban']}
def simple_model():
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                     ('m', LogisticRegression(class_weight='balanced', max_iter=2000))])

# Model pembanding dari notebook utama (dengan hyperparameter yang sama)
REF_MODELS = {k: v for k, v in MODELS.items() if k in ('Ensemble (RF + MLP + LR)', 'XGBoost', 'HistGB (pengganti XGBoost)', 'Logistic regression')}
print('Pembanding:', list(REF_MODELS), '| Model sederhana:', list(SIMPLE))

def sens_at_spec(yt, p, spec):
    f, t, _ = roc_curve(yt, p); return t[f <= 1 - spec].max()

## 1. Validasi silang berulang (fold identik dengan analisis tambahan)

In [ ]:
rows = []
t0 = time.time()
for scheme in ['individual', 'household']:
    for r in range(REPEATS):
        splits = (StratifiedGroupKFold(5, shuffle=True, random_state=SEED + r).split(X, y, groups) if scheme == 'household'
                  else StratifiedKFold(5, shuffle=True, random_state=SEED + r).split(X, y))
        for k, (tr, te) in enumerate(splits):
            fits = {name: (clone(m), FEATS) for name, m in REF_MODELS.items()}
            fits.update({name: (simple_model(), cols) for name, cols in SIMPLE.items()})
            for name, (m, cols) in fits.items():
                m.fit(X.iloc[tr][cols], y[tr]); p = m.predict_proba(X.iloc[te][cols])[:, 1]
                rows.append({'scheme': scheme, 'repeat': r, 'fold': k, 'model': name, 'n_train': len(tr), 'n_test': len(te),
                             'AUC': roc_auc_score(y[te], p), 'AUC-PR': average_precision_score(y[te], p),
                             'Brier': brier_score_loss(y[te], p),
                             'Sens@Spec0.70': sens_at_spec(y[te], p, 0.70), 'Sens@Spec0.80': sens_at_spec(y[te], p, 0.80)})
        print(f'{scheme}: selesai')
folds = pd.DataFrame(rows)
print(f'Selesai dalam {(time.time() - t0) / 60:.1f} menit')

g = folds.groupby(['scheme', 'model'])
cv_sum = g[['AUC', 'AUC-PR', 'Sens@Spec0.70', 'Sens@Spec0.80']].agg(['mean', 'std'])
cv_sum.columns = [f'{a} {b}' for a, b in cv_sum.columns]
cv_sum = cv_sum.round(3).reset_index().sort_values(['scheme', 'AUC mean'], ascending=[True, False])

def corrected_ttest(d, ref='Ensemble (RF + MLP + LR)'):
    out = []
    for scheme, ds in d.groupby('scheme'):
        piv = ds.pivot_table(index=['repeat', 'fold'], columns='model', values='AUC')
        n_te, n_tr = ds.n_test.mean(), ds.n_train.mean(); J = len(piv)
        for m in piv.columns:
            if m == ref: continue
            diff = (piv[m] - piv[ref]).values; mu, var = diff.mean(), diff.var(ddof=1)
            se = np.sqrt((1 / J + n_te / n_tr) * var); tval = mu / se if se > 0 else np.nan
            out.append({'scheme': scheme, 'model': m, 'reference': ref, 'mean AUC difference': mu,
                        '95% CI low': mu - stats.t.ppf(0.975, J - 1) * se, '95% CI high': mu + stats.t.ppf(0.975, J - 1) * se,
                        'p (corrected t-test)': 2 * stats.t.sf(abs(tval), J - 1)})
    return pd.DataFrame(out).round(4)

cv_diff = corrected_ttest(folds)
display(cv_sum); display(cv_diff)

## 2. Data uji (pembagian 80/20 yang sama dengan notebook utama)

In [ ]:
cv5 = StratifiedKFold(5, shuffle=True, random_state=SEED)
rng = np.random.default_rng(SEED)
def boot_ci(yt, p, B=1000):
    s = []
    for _ in range(B):
        i = rng.integers(0, len(yt), len(yt))
        if yt[i].min() != yt[i].max(): s.append(roc_auc_score(yt[i], p[i]))
    return np.percentile(s, [2.5, 97.5])

test_rows = []
for name, cols in SIMPLE.items():
    oof = cross_val_predict(simple_model(), X_tr[cols], y_tr, cv=cv5, method='predict_proba')[:, 1]
    f, t, th = roc_curve(y_tr, oof); thr = th[np.argmax(t - f)]
    m = simple_model().fit(X_tr[cols], y_tr); p = m.predict_proba(X_te[cols])[:, 1]; yhat = (p >= thr).astype(int)
    tp = ((yhat == 1) & (y_te == 1)).sum(); fn = ((yhat == 0) & (y_te == 1)).sum()
    tn = ((yhat == 0) & (y_te == 0)).sum(); fp = ((yhat == 1) & (y_te == 0)).sum()
    lo, hi = boot_ci(y_te, p)
    test_rows.append({'Model': name, 'Test AUC': roc_auc_score(y_te, p), '95% CI low': lo, '95% CI high': hi,
                      'AUC-PR': average_precision_score(y_te, p), 'Sensitivity': tp / (tp + fn), 'Specificity': tn / (tn + fp),
                      'Balanced accuracy': (tp / (tp + fn) + tn / (tn + fp)) / 2,
                      'Sens@Spec0.70': sens_at_spec(y_te, p, 0.70), 'Sens@Spec0.80': sens_at_spec(y_te, p, 0.80)})
    if name.startswith('LR: 5'):
        coef = pd.Series(np.exp(m.named_steps['m'].coef_[0]), index=cols).rename('OR per 1 SD (balanced weights)')
test_tab = pd.DataFrame(test_rows).round(3)
display(test_tab); display(coef.round(3).to_frame())

In [ ]:
with pd.ExcelWriter('analisis_model_sederhana.xlsx') as xw:
    cv_sum.to_excel(xw, sheet_name='RepeatedCV_summary', index=False)
    cv_diff.to_excel(xw, sheet_name='RepeatedCV_vs_ensemble', index=False)
    folds.to_excel(xw, sheet_name='RepeatedCV_folds', index=False)
    test_tab.to_excel(xw, sheet_name='Test_set', index=False)
    coef.round(3).to_frame().to_excel(xw, sheet_name='5item_OR_per_SD')
print('Tersimpan: analisis_model_sederhana.xlsx | MODE =', MODE)